# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kobeyvines/flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Answer:** This is a **signal analysis** task, not classification, clustering, or ranking.

In plain terms: I'm not trying to predict a yes/no answer for each page (that would be classification). I'm not trying to sort unlabeled pages into groups (that would be clustering). And I'm not building a "fix this page first" priority list (that would be ranking/scoring — that's a different lane's job).

What I'm actually doing: looking at a bunch of content features (word count, freshness, etc.) and checking, one at a time, whether each one is actually connected to how well a page performs. The lane is called "Ranking Signal Analysis," but the word "ranking" there means I'll rank the *signals* by how strong they are — not rank the pages.

In [ ]:
ml_task_type = "Signal analysis: measuring how strongly each content feature connects to performance"
print(ml_task_type)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Answer:** There's no single number I'm trying to predict. Instead, I'm comparing content features against a few things that were actually measured and recorded — not something I made up or calculated myself.

**What counts as "performance" here (real, measured numbers):** `impressions_90d` (how often the page showed up in search), `clicks_90d`, `sessions_90d`, `ctr` (click-through rate), `avg_position` (where it ranked), `engagement_rate`, and `scroll_rate`.

**Three things I'm deliberately NOT using as performance, and why:**
- `search_volume` — this just means "how many people search for this topic." It tells me about demand, not about how well the page actually did. (We already saw in w01 that it barely correlates with impressions — 0.0012 — so treating it as performance would be misleading.)
- `trend_direction` — this is a label someone else already built from the data ("is this page trending down?"). Using it would mean I'm just checking whether my features predict *their* summary, not the real numbers underneath. It also belongs to a different lane (Lane 2), not this one.
- `ai_traffic_pct` — very few pages have any AI-referral data at all, so any pattern I find here would likely be noise, not a real signal.

In [ ]:
# The real, measured outcomes I will compare features against:
observed_outcome_metrics = [
    "impressions_90d", "clicks_90d", "sessions_90d",
    "ctr", "avg_position", "engagement_rate", "scroll_rate",
]

# Fields I'm choosing NOT to treat as "performance", and why:
excluded_from_target_role = {
    "search_volume": "measures demand, not how the page actually performed",
    "trend_direction": "already a summary someone else built -- and it belongs to Lane 2",
    "ai_traffic_pct": "too few pages have this data to trust any pattern in it",
}

print("Comparing features against:", observed_outcome_metrics)
print("\nNot using as performance:")
for k, v in excluded_from_target_role.items():
    print(f"  - {k}: {v}")


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Answer:** For each feature, I'll measure how strongly it moves together with each performance number above. In plain terms: does this feature go up when performance goes up (or down)? That's called a **correlation** — a number between -1 and 1 that says how tightly two things move together. 0 means no connection, 1 or -1 means a very tight connection.

**One important catch:** I won't just compare all 30,000 pages against each other directly. Pages that already rank #2 in search naturally get way more clicks than pages ranking #18 — that's not because of my feature, that's just position. So I'll compare pages only against other pages in a similar spot (same `position_tier`, same `content_type`), so I'm not fooled into thinking position's effect is my feature's effect.

**What counts as "good enough to matter":** the correlation has to be at least 0.10 in either direction, it has to show up consistently (not just by luck), and the direction (up or down) has to hold for most of the 32 clients — not just one or two. I'm setting the bar this high on purpose: with 30,000 rows, even a tiny, meaningless correlation like 0.02 can look "real" on paper. w01 already caught exactly that trap once.

In [ ]:
success_metric = (
    "Correlation strength (Spearman rho) between each feature and each performance metric, "
    "measured within similar pages (same position tier & content type). "
    "Needs: |rho| >= 0.10, holds up statistically, and same direction for most of the 32 clients."
)
print(success_metric)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Answer:** One row = one content page. Each page has its own ID (`content_id`), and it belongs to one client (`client_id`). `client_id` is just a way to group pages by who they belong to — it is NOT what one row represents. Before trusting anything else, I check that each `content_id` really only shows up once — otherwise a duplicated page could quietly throw off every number I calculate later.

In [ ]:
import os
import pandas as pd

candidate_paths = [
    "data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
]
file_path = next((p for p in candidate_paths if os.path.exists(p)), None)
if file_path is None:
    raise FileNotFoundError("Could not find the dataset in the expected locations.")

df = pd.read_csv(file_path)

print(f"Loaded from: {file_path}")
print(f"Rows = pages, columns = features. Shape: {df.shape}")
print(f"Is every content_id unique? {df['content_id'].is_unique}")
print(f"All columns: {df.columns.tolist()}")
df.head()


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Answer:** A simple rule like "pages over 2,000 words do better" sounds reasonable, but it breaks in three ways:

1. **Features are tangled together.** Longer pages might also happen to be older, or from bigger clients, or cover different topics. A simple rule can't tell which of those things is actually doing the work.
2. **What's true for one client isn't true for another.** The code below checks this directly — I look at the word-count-vs-performance relationship separately for each of the 32 clients, and see how often it points the same direction.
3. **A drop in numbers doesn't always mean what it looks like.** The dataset guide calls this out directly: a page's traffic can fall because of consolidation (a sister page took over), seasonality (fewer people search for it right now), or plain random noise — not because of anything about the content itself. A fixed rule can't tell these apart. A careful, feature-by-feature, client-by-client comparison at least has a chance of catching it.

In [ ]:
required = {"word_count", "impressions_90d", "client_id"}
if required.issubset(df.columns):
    # For each client, how strongly does word_count line up with impressions_90d?
    per_client_corr = (
        df.groupby("client_id")
          .apply(lambda g: g["word_count"].corr(g["impressions_90d"], method="spearman"))
    )
    print("Word count vs impressions, correlation per client:")
    print(per_client_corr.describe())

    pct_positive = (per_client_corr > 0).mean() * 100
    print(f"\n{pct_positive:.1f}% of clients: longer pages do better.")
    print(f"{100 - pct_positive:.1f}% of clients: no relationship, or the opposite.")
    print("That split is exactly why one fixed rule can't work for everyone.")
else:
    missing = required - set(df.columns)
    print(f"Missing columns: {missing} -- check df.columns above first.")


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)  <!-- run once locally to confirm column names -->
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.